In [1]:
from src.data_preprocessing.TextTokenizer import TextTokenizer
from src.data_preprocessing.TextEmbedder import TextEmbedder
from src.data_preprocessing.USDA_processing import embed_USDA_data
import src.config as config
from src.util.data_ingestion import ingest_to_supabase, add_metadata_tags
from src.util.supabase_utils import connect_supabase, query_embeddings

2026-10-10 17:15:20.529998: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-10 17:15:20.538545: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791666920.548089  309088 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791666920.551091  309088 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791666920.559101  309088 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
embedder= TextEmbedder("sentence-transformers/all-MiniLM-L6-v2")

CHUNK_METHOD= "word" # word or sentence
CHUNK_MAX_SIZE= 192
CHUNK_OVERLAP_AMOUNT= 48

records= embed_USDA_data(
    config.Data.USDA.ROOT,
    embedder,
    CHUNK_METHOD,
    CHUNK_MAX_SIZE,
    CHUNK_OVERLAP_AMOUNT
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error:

In [3]:
client = connect_supabase()

In [4]:
ingest_to_supabase(records, client)

17:16:26 [INFO    ]: Inserted 1000 embeddings to supabase
17:16:30 [INFO    ]: Inserted 2000 embeddings to supabase
17:16:36 [INFO    ]: Inserted 3000 embeddings to supabase
17:16:39 [INFO    ]: Inserted 4000 embeddings to supabase
17:16:42 [INFO    ]: Inserted 5000 embeddings to supabase
17:16:45 [INFO    ]: Inserted 6000 embeddings to supabase
17:16:50 [INFO    ]: Inserted 7000 embeddings to supabase
17:16:52 [INFO    ]: Inserted 7692 embeddings to supabase
17:16:52 [INFO    ]: Completed: Inserted 7692 embeddings to supabase


In [4]:
#add_metadata_tags(records, client)

17:07:42 [INFO    ]: Updated 1915 plant codes


In [6]:
query_text = "how to treat leaf spot on fir trees"
query_embedding = embedder.encode(query_text).tolist()
query_embeddings(
    client,
    query_embedding,
    plant_code="ABAM",
    match_threshold=.5
)

[{'id': 7,
  'title': 'ABAM',
  'body': 'foliage, stem decay, and abortive cones. The most reliable way to detect this disease is by the presence of fruiting bodies in the duff layer at the root collar on the outer bark. Trees become infected by rood contact or by airborne spores falling onto woonds. Other fungal diseases include Indian paint fungus (Echinodontium tinctorium) and laminated root rot (Phellinus weirii). Both can infect stands of trees and result in patches of damaged or dead trees. Trees weakened by disease or poor growing conditions may become infested with fir-engraver beetle (Scolylus ventralis), silver fir beetle (Pseudohylesinus sericeus), or fir root bark beetle (Pseudohylesinus granulatus). In large numbers, these beetles may kill entire trees before any symptoms are observed. Pacific silver firs are also susceptible to Western spruce budworm (Choristeneura',
  'plant_code': 'ABAM',
  'common_name': 'Pacific silver fir',
  'scientific_name': 'Abies amabilis (Dougl

In [7]:
client = connect_supabase()
print(client.table("embeddings").select("id", count="exact").execute().count)
rows = client.table("embeddings").select("plant_code").execute().data
print(len({r["plant_code"] for r in rows}))

7692
73
